In [ ]:
!pip install cartopy
import cartopy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 76.3 MB/s eta 0:00:00


In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs # Import cartopy.crs here
import cartopy.feature as cfeature

In [ ]:
import xarray as xr
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import os
from datetime import datetime
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import xarray as xr
file_path = "/content/drive/MyDrive/era5_vn_min/era5_vn_2m_dewpoint_temperature_2020_2024.nc"
ds = xr.open_dataset(file_path)

# Khám phá nhanh dataset ERA5 d2m

- **Dimensions:**  
  - `valid_time`: 7308 (6h/lần từ 2020-01-01 → 2024-12-31)  
  - `latitude`: 65 (24° → 8°N)  
  - `longitude`: 33 (102° → 110°E)  

- **Data variable chính:**  
  - `d2m` (valid_time × latitude × longitude) – điểm sương 2m  

- **Coordinates phụ:**  
  - `expver`, `number` – metadata, ít quan trọng cho mô hình  

- **Mục tiêu tiếp theo:**  
  1. Chuyển `d2m` thành tensor PyTorch `(time, lat, lon, features=1)`  
  2. Chuẩn hóa dữ liệu (mean-std)  
  3. Tạo các sequence theo thời gian (window) cho transformer  
  4. Chia train/val/test


In [ ]:
ds

<xarray.Dataset> Size: 63MB
Dimensions:     (valid_time: 7308, latitude: 65, longitude: 33)
Coordinates:
    number      int64 8B ...
  * valid_time  (valid_time) datetime64[ns] 58kB 2020-01-01 ... 2024-12-31T18...
  * latitude    (latitude) float64 520B 24.0 23.75 23.5 23.25 ... 8.5 8.25 8.0
  * longitude   (longitude) float64 264B 102.0 102.2 102.5 ... 109.5 109.8 110.0
    expver      (valid_time) <U4 117kB ...
Data variables:
    d2m         (valid_time, latitude, longitude) float32 63MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-10-03T05:55 GRIB to CDM+CF via cfgrib-0.9.1...

In [ ]:
print("Thời điểm đầu/cuối:", ds['valid_time'].values[0], "→", ds['valid_time'].values[-1])

Thời điểm đầu/cuối: 2020-01-01T00:00:00.000000000 → 2024-12-31T18:00:00.000000000


In [ ]:
ds['d2m'].attrs

{'GRIB_paramId': np.int64(168),
 'GRIB_dataType': 'an',
 'GRIB_numberOfPoints': np.int64(2145),
 'GRIB_typeOfLevel': 'surface',
 'GRIB_stepUnits': np.int64(1),
 'GRIB_stepType': 'instant',
 'GRIB_gridType': 'regular_ll',
 'GRIB_uvRelativeToGrid': np.int64(0),
 'GRIB_NV': np.int64(0),
 'GRIB_Nx': np.int64(33),
 'GRIB_Ny': np.int64(65),
 'GRIB_cfName': 'unknown',
 'GRIB_cfVarName': 'd2m',
 'GRIB_gridDefinitionDescription': 'Latitude/Longitude Grid',
 'GRIB_iDirectionIncrementInDegrees': np.float64(0.25),
 'GRIB_iScansNegatively': np.int64(0),
 'GRIB_jDirectionIncrementInDegrees': np.float64(0.25),
 'GRIB_jPointsAreConsecutive': np.int64(0),
 'GRIB_jScansPositively': np.int64(0),
 'GRIB_latitudeOfFirstGridPointInDegrees': np.float64(24.0),
 'GRIB_latitudeOfLastGridPointInDegrees': np.float64(8.0),
 'GRIB_longitudeOfFirstGridPointInDegrees': np.float64(102.0),
 'GRIB_longitudeOfLastGridPointInDegrees': np.float64(110.0),
 'GRIB_missingValue': np.float64(3.4028234663852886e+38),
 'GRIB_name

- **Ý nghĩa:**  
  Mỗi giá trị là điểm sương ở **chiều cao 2 mét**, tức là độ ẩm tương đối “chứa trong không khí” tại mặt đất.

- **Units:** K (Kelvin)  
  Dữ liệu nhiệt độ được lưu theo Kelvin. Nếu muốn dùng cho mô hình, có thể **chuẩn hóa (mean-std)** hoặc đổi sang Celsius tùy mục đích.

- **Dtype:** float32 → convert sang PyTorch `.float()`  
  Để đảm bảo **tính tương thích với mô hình và hàm loss** trong PyTorch.


**ERA5 đã nội suy và điền các missing value**

In [ ]:
ds.isnull().sum()

<xarray.Dataset> Size: 16B
Dimensions:  ()
Coordinates:
    number   int64 8B ...
Data variables:
    d2m      int64 8B 0

In [ ]:
import numpy as np

# Giả sử da là xarray.DataArray
da = ds['d2m']

# Lấy giá trị missing từ attributes nếu có
missing_val = da.attrs.get('GRIB_missingValue', None)

if missing_val is not None:
    num_missing = np.sum(da.values == missing_val)
    print(f"Số giá trị missing (GRIB_missingValue): {num_missing}")
else:
    print("Không tìm thấy GRIB_missingValue")


Số giá trị missing (GRIB_missingValue): 0


In [ ]:
ds['d2m'].shape

(7308, 65, 33)

In [ ]:
ds['d2m'].dims

('valid_time', 'latitude', 'longitude')

In [ ]:
# giá trị ngày đầu tiên
ds['d2m'].sel(valid_time='2020-01-01T00:00:00')

<xarray.DataArray 'd2m' (latitude: 65, longitude: 33)> Size: 9kB
array([[282.73877, 282.1919 , 282.17432, ..., 281.90088, 283.5376 , 282.938  ],
       [282.7583 , 282.76807, 283.0786 , ..., 283.2329 , 283.77197, 283.29736],
       [282.52197, 283.7212 , 283.52588, ..., 283.51807, 283.62354, 283.6997 ],
       ...,
       [295.54346, 295.46924, 295.3833 , ..., 297.0337 , 297.15674, 297.24854],
       [295.729  , 295.70166, 295.63916, ..., 297.00635, 297.18213, 297.2798 ],
       [295.8462 , 295.84814, 295.85596, ..., 296.96924, 297.1079 , 297.13525]],
      dtype=float32)
Coordinates:
    number      int64 8B ...
    valid_time  datetime64[ns] 8B 2020-01-01
  * latitude    (latitude) float64 520B 24.0 23.75 23.5 23.25 ... 8.5 8.25 8.0
  * longitude   (longitude) float64 264B 102.0 102.2 102.5 ... 109.5 109.8 110.0
    expver      <U4 16B ...
Attributes: (12/32)
    GRIB_paramId:                             168
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      2145
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_totalNumber:                         0
    GRIB_units:                               K
    long_name:                                2 metre dewpoint temperature
    units:                                    K
    standard_name:                            unknown
    GRIB_surface:                             0.0

In [ ]:
ds['d2m'].isel(valid_time=0)  # snapshot đầu tiên

<xarray.DataArray 'd2m' (latitude: 65, longitude: 33)> Size: 9kB
array([[282.73877, 282.1919 , 282.17432, ..., 281.90088, 283.5376 , 282.938  ],
       [282.7583 , 282.76807, 283.0786 , ..., 283.2329 , 283.77197, 283.29736],
       [282.52197, 283.7212 , 283.52588, ..., 283.51807, 283.62354, 283.6997 ],
       ...,
       [295.54346, 295.46924, 295.3833 , ..., 297.0337 , 297.15674, 297.24854],
       [295.729  , 295.70166, 295.63916, ..., 297.00635, 297.18213, 297.2798 ],
       [295.8462 , 295.84814, 295.85596, ..., 296.96924, 297.1079 , 297.13525]],
      dtype=float32)
Coordinates:
    number      int64 8B ...
    valid_time  datetime64[ns] 8B 2020-01-01
  * latitude    (latitude) float64 520B 24.0 23.75 23.5 23.25 ... 8.5 8.25 8.0
  * longitude   (longitude) float64 264B 102.0 102.2 102.5 ... 109.5 109.8 110.0
    expver      <U4 16B ...
Attributes: (12/32)
    GRIB_paramId:                             168
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      2145
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_totalNumber:                         0
    GRIB_units:                               K
    long_name:                                2 metre dewpoint temperature
    units:                                    K
    standard_name:                            unknown
    GRIB_surface:                             0.0

In [ ]:
print(ds['d2m'].min().values)
print(ds['d2m'].max().values)

260.331787109375
303.971923828125


In [ ]:
# ví dụ 3 lat × 3 lon đầu tiên của thời điểm đầu
ds['d2m'].isel(valid_time=0, latitude=slice(0,3), longitude=slice(0,3))


<xarray.DataArray 'd2m' (latitude: 3, longitude: 3)> Size: 36B
array([[282.73877, 282.1919 , 282.17432],
       [282.7583 , 282.76807, 283.0786 ],
       [282.52197, 283.7212 , 283.52588]], dtype=float32)
Coordinates:
    number      int64 8B ...
    valid_time  datetime64[ns] 8B 2020-01-01
  * latitude    (latitude) float64 24B 24.0 23.75 23.5
  * longitude   (longitude) float64 24B 102.0 102.2 102.5
    expver      <U4 16B ...
Attributes: (12/32)
    GRIB_paramId:                             168
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      2145
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_totalNumber:                         0
    GRIB_units:                               K
    long_name:                                2 metre dewpoint temperature
    units:                                    K
    standard_name:                            unknown
    GRIB_surface:                             0.0

In [ ]:
small_ds = ds['d2m'].isel(valid_time=slice(0, 5), latitude=slice(0, 3), longitude=slice(0, 3))
df = small_ds.to_dataframe().reset_index()  # Chuyển sang DataFrame
print("\nMẫu table (5 timesteps đầu, 3 lat đầu, 3 lon đầu):")
df.head(20)  # Print 20 dòng đầu của table (vì có multi-index ban đầu)


Mẫu table (5 timesteps đầu, 3 lat đầu, 3 lon đầu):


,valid_time,latitude,longitude,number,expver,d2m
0,2020-01-01 00:00:00,24.00,102.00,0,0001,282.738770
1,2020-01-01 00:00:00,24.00,102.25,0,0001,282.191895
2,2020-01-01 00:00:00,24.00,102.50,0,0001,282.174316
3,2020-01-01 00:00:00,23.75,102.00,0,0001,282.758301
4,2020-01-01 00:00:00,23.75,102.25,0,0001,282.768066
5,2020-01-01 00:00:00,23.75,102.50,0,0001,283.078613
6,2020-01-01 00:00:00,23.50,102.00,0,0001,282.521973
7,2020-01-01 00:00:00,23.50,102.25,0,0001,283.721191
8,2020-01-01 00:00:00,23.50,102.50,0,0001,283.525879
9,2020-01-01 06:00:00,24.00,102.00,0,0001,283.522217


merge file

In [ ]:
import xarray as xr

# Load từng file
d2m  = xr.open_dataset("/content/drive/MyDrive/era5_vn_min/era5_vn_2m_dewpoint_temperature_2020_2024.nc")
u10  = xr.open_dataset("/content/drive/MyDrive/era5_vn_min/era5_vn_10m_u_component_of_wind_2020_2024.nc")
v10  = xr.open_dataset("/content/drive/MyDrive/era5_vn_min/era5_vn_10m_v_component_of_wind_2020_2024.nc")
tcwv = xr.open_dataset("/content/drive/MyDrive/era5_vn_min/era5_vn_total_column_water_vapour_2020_2024.nc")
tp   = xr.open_dataset("/content/drive/MyDrive/era5_vn_min/era5_vn_total_precipitation_2020_2024.nc")

# Merge theo trục thời gian - lat - lon
ds_all = xr.merge([d2m, u10, v10, tcwv, tp])
ds_all


/tmp/ipython-input-1947832726.py:11: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_all = xr.merge([d2m, u10, v10, tcwv, tp])
/tmp/ipython-input-1947832726.py:11: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_all = xr.merge([d2m, u10, v10, tcwv, tp])


<xarray.Dataset> Size: 314MB
Dimensions:     (valid_time: 7308, latitude: 65, longitude: 33)
Coordinates:
    number      int64 8B 0
  * valid_time  (valid_time) datetime64[ns] 58kB 2020-01-01 ... 2024-12-31T18...
  * latitude    (latitude) float64 520B 24.0 23.75 23.5 23.25 ... 8.5 8.25 8.0
  * longitude   (longitude) float64 264B 102.0 102.2 102.5 ... 109.5 109.8 110.0
    expver      (valid_time) <U4 117kB '0001' '0001' '0001' ... '0001' '0001'
Data variables:
    d2m         (valid_time, latitude, longitude) float32 63MB ...
    u10         (valid_time, latitude, longitude) float32 63MB ...
    v10         (valid_time, latitude, longitude) float32 63MB ...
    tcwv        (valid_time, latitude, longitude) float32 63MB ...
    tp          (valid_time, latitude, longitude) float32 63MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-10-03T05:55 GRIB to CDM+CF via cfgrib-0.9.1...

In [ ]:
# Lưu file
save_path = "/content/drive/MyDrive/era5_vn_min/era5_vn_merged_2020_2024.nc"
ds_all.to_netcdf(save_path)


In [ ]:
ds_all = xr.open_dataset("/content/drive/MyDrive/era5_vn_min/era5_vn_merged_2020_2024.nc")
ds_all


<xarray.Dataset> Size: 314MB
Dimensions:     (valid_time: 7308, latitude: 65, longitude: 33)
Coordinates:
    number      int64 8B ...
  * valid_time  (valid_time) datetime64[ns] 58kB 2020-01-01 ... 2024-12-31T18...
  * latitude    (latitude) float64 520B 24.0 23.75 23.5 23.25 ... 8.5 8.25 8.0
  * longitude   (longitude) float64 264B 102.0 102.2 102.5 ... 109.5 109.8 110.0
    expver      (valid_time) <U4 117kB ...
Data variables:
    d2m         (valid_time, latitude, longitude) float32 63MB ...
    u10         (valid_time, latitude, longitude) float32 63MB ...
    v10         (valid_time, latitude, longitude) float32 63MB ...
    tcwv        (valid_time, latitude, longitude) float32 63MB ...
    tp          (valid_time, latitude, longitude) float32 63MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-10-03T05:55 GRIB to CDM+CF via cfgrib-0.9.1...

xem thử table 1 ngày

In [ ]:
df_sample = ds_all.sel(valid_time="2020-01-01").to_dataframe().reset_index()
df_sample.head()


,valid_time,latitude,longitude,d2m,u10,v10,tcwv,tp,number,expver
0,2020-01-01,24.0,102.00,282.738770,-0.549927,0.168015,17.670929,1.144409e-05,0,0001
1,2020-01-01,24.0,102.25,282.191895,0.146362,0.737350,16.329132,1.144409e-05,0,0001
2,2020-01-01,24.0,102.50,282.174316,-0.005005,0.916061,15.430696,9.536743e-07,0,0001
3,2020-01-01,24.0,102.75,282.396973,0.309448,0.759811,15.448274,0.000000e+00,0,0001
4,2020-01-01,24.0,103.00,283.096191,0.680542,0.767624,16.272491,0.000000e+00,0,0001


In [ ]:
for var in ds_all.data_vars:
    print(var, ":", ds_all[var].attrs.get("units", "no_units"))


d2m : K
u10 : m s**-1
v10 : m s**-1
tcwv : kg m**-2
tp : m


In [ ]:
for var in ds_all.data_vars:
    print(var, ds_all[var].shape, ds_all[var].dims)


d2m (7308, 65, 33) ('valid_time', 'latitude', 'longitude')
u10 (7308, 65, 33) ('valid_time', 'latitude', 'longitude')
v10 (7308, 65, 33) ('valid_time', 'latitude', 'longitude')
tcwv (7308, 65, 33) ('valid_time', 'latitude', 'longitude')
tp (7308, 65, 33) ('valid_time', 'latitude', 'longitude')


In [ ]:
ds_all.isnull().sum()

<xarray.Dataset> Size: 48B
Dimensions:  ()
Coordinates:
    number   int64 8B ...
Data variables:
    d2m      int64 8B 0
    u10      int64 8B 0
    v10      int64 8B 0
    tcwv     int64 8B 0
    tp       int64 8B 0

In [ ]:
ds_all.data_vars

Data variables:
    d2m      (valid_time, latitude, longitude) float32 63MB 282.7 ... 297.0
    u10      (valid_time, latitude, longitude) float32 63MB -0.5499 ... -0.8317
    v10      (valid_time, latitude, longitude) float32 63MB 0.168 ... -7.399
    tcwv     (valid_time, latitude, longitude) float32 63MB 17.67 ... 63.42
    tp       (valid_time, latitude, longitude) float32 63MB 1.144e-05 ... 6.0...

In [ ]:
print("Thời điểm đầu/cuối:", ds['valid_time'].values[0], "→", ds['valid_time'].values[-1])


Thời điểm đầu/cuối: 2020-01-01T00:00:00.000000000 → 2024-12-31T18:00:00.000000000


xử lí chuẩn hóa thoi

In [ ]:
import pandas as pd

stats = {}
for var in ds_all.data_vars:
    da = ds_all[var]
    stats[var] = {
        "units": da.attrs.get("units", "no_units"),
        "min": float(da.min().values),
        "max": float(da.max().values),
        "mean": float(da.mean().values),
        "std": float(da.std().values),
    }

df_stats = pd.DataFrame.from_dict(stats, orient="index")
df_stats


,units,min,max,mean,std
d2m,K,260.331787,303.971924,265.889069,29.459312
u10,m s**-1,-25.107941,23.230942,-0.595049,2.894487
v10,m s**-1,-26.736282,23.074432,-0.052967,2.955371
tcwv,kg m**-2,3.561975,87.785645,47.416645,13.718238
tp,m,0.000000,0.041861,0.000212,0.000681


In [ ]:
import numpy as np

# Copy dataset để tránh ghi đè gốc
ds_norm = ds_all.copy()

# Đổi đơn vị từ m -> mm
ds_norm['tp'] = ds_norm['tp'] * 1000
ds_norm['tp'].attrs['units'] = "mm"  # cập nhật đơn vị

# Log-transform để giảm skew
ds_norm['tp'] = np.log1p(ds_norm['tp'])
ds_norm['tp'].attrs['transform'] = "log1p"

In [ ]:
import json

norm_stats = {}

for var in ['d2m', 'u10', 'v10', 'tcwv', 'tp']:
    # Lấy metadata gốc trước khi chuẩn hoá
    units = ds_all[var].attrs.get("units", "no_units")
    transform = "z-score"
    if var == "tp":
        units = "mm"   # vì đã đổi m -> mm
        transform = "log1p + z-score"

    # Tính mean/std trên dữ liệu đã chuyển đổi (ds_norm)
    mean = float(ds_norm[var].mean().values)
    std = float(ds_norm[var].std().values)

    # Chuẩn hoá inplace
    ds_norm[var] = (ds_norm[var] - mean) / std

    # Lưu stats lại
    norm_stats[var] = {
        "mean": mean,
        "std": std,
        "units": units,
        "transform": transform
    }

    print(f"{var} -> mean={mean:.3f}, std={std:.3f}, units={units}, transform={transform}")

# Lưu file JSON
with open("norm_stats.json", "w") as f:
    json.dump(norm_stats, f, indent=4)


d2m -> mean=0.966, std=0.154, units=K, transform=z-score
u10 -> mean=-0.000, std=1.000, units=m s**-1, transform=z-score
v10 -> mean=0.000, std=1.000, units=m s**-1, transform=z-score
tcwv -> mean=-0.169, std=0.990, units=kg m**-2, transform=z-score
tp -> mean=0.006, std=0.992, units=mm, transform=log1p + z-score


In [ ]:
# Tạo encoding chỉ bật zlib, giữ nguyên dtype
encoding = {var: {'zlib': True, 'complevel': 4} for var in ds_norm.data_vars}

# Lưu NetCDF vào Google Drive
save_path = "/content/drive/MyDrive/era5_vn_min/era5_vn_merged_normalized_2020_2024.nc"
ds_norm.to_netcdf(save_path, encoding=encoding)

In [ ]:
# Chia dataset theo time
ds_train = ds_norm.sel(valid_time=slice("2020-01-01", "2022-12-31"))
ds_val   = ds_norm.sel(valid_time=slice("2023-01-01", "2023-12-31"))
ds_test  = ds_norm.sel(valid_time=slice("2024-01-01", "2024-12-31"))

# Kiểm tra shape từng phần
print("Train:", ds_train['d2m'].shape)
print("Validation:", ds_val['d2m'].shape)
print("Test:", ds_test['d2m'].shape)

Train: (4384, 65, 33)
Validation: (1460, 65, 33)
Test: (1464, 65, 33)


In [ ]:
# Lưu từng file NetCDF
ds_train.to_netcdf("/content/drive/MyDrive/era5_vn_min/train_2020_2022.nc")
ds_val.to_netcdf("/content/drive/MyDrive/era5_vn_min/val_2023.nc")
ds_test.to_netcdf("/content/drive/MyDrive/era5_vn_min/test_2024.nc")

In [ ]:
# Đường dẫn file NetCDF
train_path = "/content/drive/MyDrive/era5_vn_min/train_2020_2022.nc"
val_path   = "/content/drive/MyDrive/era5_vn_min/val_2023.nc"
test_path  = "/content/drive/MyDrive/era5_vn_min/test_2024.nc"

# Load dataset
ds_train = xr.open_dataset(train_path)
ds_val   = xr.open_dataset(val_path)
ds_test  = xr.open_dataset(test_path)

In [ ]:
# Kiểm tra min/max các biến
for ds, name in zip([ds_train, ds_val, ds_test], ["Train", "Validation", "Test"]):
    print(f"\n{name} min/max per variable:")
    for var in ds.data_vars:
        print(f"{var}: min={ds[var].min().values:.3f}, max={ds[var].max().values:.3f}")


Train min/max per variable:
d2m: min=-7.126, max=1.915
u10: min=-8.138, max=7.857
v10: min=-8.828, max=7.467
tcwv: min=-2.994, max=2.735
tp: min=-0.457, max=12.750

Validation min/max per variable:
d2m: min=-7.480, max=1.778
u10: min=-4.785, max=6.661
v10: min=-6.186, max=4.684
tcwv: min=-3.060, max=3.144
tp: min=-0.457, max=11.784

Test min/max per variable:
d2m: min=-7.444, max=2.116
u10: min=-8.473, max=8.235
v10: min=-9.032, max=7.828
tcwv: min=-2.960, max=3.016
tp: min=-0.457, max=11.418


In [ ]:
# Kiểm tra thông tin dataset
print("Train:")
print(ds_train)
print("\nValidation:")
print(ds_val)
print("\nTest:")
print(ds_test)


Train:
<xarray.Dataset> Size: 188MB
Dimensions:     (valid_time: 4384, latitude: 65, longitude: 33)
Coordinates:
    number      int64 8B ...
  * valid_time  (valid_time) datetime64[ns] 35kB 2020-01-01 ... 2022-12-31T18...
  * latitude    (latitude) float64 520B 24.0 23.75 23.5 23.25 ... 8.5 8.25 8.0
  * longitude   (longitude) float64 264B 102.0 102.2 102.5 ... 109.5 109.8 110.0
    expver      (valid_time) <U4 70kB ...
Data variables:
    d2m         (valid_time, latitude, longitude) float32 38MB -2.553 ... 0.4447
    u10         (valid_time, latitude, longitude) float32 38MB 0.01561 ... 0....
    v10         (valid_time, latitude, longitude) float32 38MB 0.07479 ... -4...
    tcwv        (valid_time, latitude, longitude) float32 38MB -2.021 ... 1.025
    tp          (valid_time, latitude, longitude) float32 38MB -0.4173 ... -0...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          